
## LangChain + ChromaDB: Building a RAG Pipeline


In [14]:
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_ollama import ChatOllama

In [15]:

loader = TextLoader("agriculture.txt")

documents = loader.load()

print(documents)

[Document(metadata={'source': 'agriculture.txt'}, page_content='Soil Health and Fertilizer Management\n\nSoil pH is an important factor in plant growth.\nMost crops grow well when soil pH is between 6.5 and 7.5.\n\nNitrogen is an essential nutrient for plant growth.\nIt helps plants develop healthy leaves and stems.\n\nPhosphorus supports root development and flowering.\nPotassium helps plants improve disease resistance and overall strength.\n\nOrganic matter improves soil structure and water retention.\nAdding compost can increase organic matter in the soil.')]


In [16]:

splitter = RecursiveCharacterTextSplitter( chunk_size=300, chunk_overlap=50 )

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    
    print(f"\nChunk {i+1}:")
    
    print(chunk.page_content)
    

Number of chunks: 2

Chunk 1:
Soil Health and Fertilizer Management

Soil pH is an important factor in plant growth.
Most crops grow well when soil pH is between 6.5 and 7.5.

Nitrogen is an essential nutrient for plant growth.
It helps plants develop healthy leaves and stems.

Chunk 2:
Phosphorus supports root development and flowering.
Potassium helps plants improve disease resistance and overall strength.

Organic matter improves soil structure and water retention.
Adding compost can increase organic matter in the soil.


In [18]:

embedding = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2" )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3858.19it/s]


### STORE CHUNKS IN CHROMADB

In [19]:

vectorstore = Chroma.from_documents(
    
    documents=chunks,
    
    embedding=embedding,
    
    persist_directory="chroma_db"
)


###  SIMILARITY SEARCH

In [20]:

query = "What is the ideal soil pH?"

docs = vectorstore.similarity_search( query, k=2 )

for doc in docs:
    
    print("\nRetrieved:")
    
    print(doc.page_content)



Retrieved:
Soil Health and Fertilizer Management

Soil pH is an important factor in plant growth.
Most crops grow well when soil pH is between 6.5 and 7.5.

Nitrogen is an essential nutrient for plant growth.
It helps plants develop healthy leaves and stems.

Retrieved:
Soil Health and Fertilizer Management

Soil pH is an important factor in plant growth.
Most crops grow well when soil pH is between 6.5 and 7.5.

Nitrogen is an essential nutrient for plant growth.
It helps plants develop healthy leaves and stems.


### SIMILARITY SEARCH WITH SCORES

In [21]:

docs_with_scores = vectorstore.similarity_search_with_score( query, k=3)

for doc, score in docs_with_scores:
    
    print("\nScore:", score)
    
    print("Retrieved:")
    
    print(doc.page_content)



Score: 0.6709783673286438
Retrieved:
Soil Health and Fertilizer Management

Soil pH is an important factor in plant growth.
Most crops grow well when soil pH is between 6.5 and 7.5.

Nitrogen is an essential nutrient for plant growth.
It helps plants develop healthy leaves and stems.

Score: 0.6709783673286438
Retrieved:
Soil Health and Fertilizer Management

Soil pH is an important factor in plant growth.
Most crops grow well when soil pH is between 6.5 and 7.5.

Nitrogen is an essential nutrient for plant growth.
It helps plants develop healthy leaves and stems.

Score: 1.0481716394424438
Retrieved:
Phosphorus supports root development and flowering.
Potassium helps plants improve disease resistance and overall strength.

Organic matter improves soil structure and water retention.
Adding compost can increase organic matter in the soil.


### CREATE RETRIEVER

In [22]:

retriever = vectorstore.as_retriever( search_kwargs={"k": 2} )
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001FF41A25DC0>, search_kwargs={'k': 2})

###  TEST RETRIEVER

In [23]:

docs = retriever.invoke(query)

for doc in docs:
    print("\nRetrieved:")
    print(doc.page_content)


Retrieved:
Soil Health and Fertilizer Management

Soil pH is an important factor in plant growth.
Most crops grow well when soil pH is between 6.5 and 7.5.

Nitrogen is an essential nutrient for plant growth.
It helps plants develop healthy leaves and stems.

Retrieved:
Soil Health and Fertilizer Management

Soil pH is an important factor in plant growth.
Most crops grow well when soil pH is between 6.5 and 7.5.

Nitrogen is an essential nutrient for plant growth.
It helps plants develop healthy leaves and stems.


### INTRODUCE LLM

In [24]:

llm = ChatOllama(  model="llama3.2", temperature=0 )

response = llm.invoke( "What is soil pH?")

print(response.content)


Soil pH is a measure of the acidity or alkalinity of the soil. It is a scale that ranges from 0 to 14, with 7 being neutral. The pH scale is as follows:

* Acidic: 0-6.9 (low pH)
* Neutral: 7 (neutral pH)
* Alkaline: 7.1-14 (high pH)

Soil pH is an important factor in determining the availability of nutrients for plants, as well as the presence of contaminants and pathogens. Here's how pH affects plants:

* Acidic soils (pH < 6.9) can be challenging for plants, as they may not be able to absorb essential nutrients like nitrogen and phosphorus.
* Neutral soils (pH 7) are generally considered optimal for plant growth, as they provide a balanced range of nutrients.
* Alkaline soils (pH > 7) can be beneficial for plants that thrive in high-pH conditions, such as some types of grasses and legumes.

Soil pH can be influenced by various factors, including:

* Parent material: The type of rocks and minerals present in the soil can affect its pH.
* Organic matter: The amount and type of organic

### COMBINE RETRIEVAL + LLM

In [25]:

question = "What is the ideal soil pH?"

docs = retriever.invoke(question)

context = "\n\n".join( doc.page_content for doc in docs )

prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""


In [26]:
prompt 

'\nAnswer the question using ONLY the context below.\n\nContext:\nSoil Health and Fertilizer Management\n\nSoil pH is an important factor in plant growth.\nMost crops grow well when soil pH is between 6.5 and 7.5.\n\nNitrogen is an essential nutrient for plant growth.\nIt helps plants develop healthy leaves and stems.\n\nSoil Health and Fertilizer Management\n\nSoil pH is an important factor in plant growth.\nMost crops grow well when soil pH is between 6.5 and 7.5.\n\nNitrogen is an essential nutrient for plant growth.\nIt helps plants develop healthy leaves and stems.\n\nQuestion:\nWhat is the ideal soil pH?\n\nAnswer:\n'

In [27]:

response = llm.invoke(prompt)

print(response.content)

The ideal soil pH is between 6.5 and 7.5.


### Nodes

#### persist_directory="chroma_db"

    This tells ChromaDB where to save the vector database on your computer.

    "ChromaDB, save my database inside a folder called chroma_db."

### similarity_search() vs similarity_search_with_score()

    Both methods search ChromaDB for documents similar to the query.

The main difference is:

    similarity_search() → returns documents
    
    similarity_search_with_score() → returns documents + similarity/distance score
    

| Method                           | Returns           | Use when                                   |
| -------------------------------- | ----------------- | ------------------------------------------ |
| `similarity_search()`            | Documents         | You only need relevant documents           |
| `similarity_search_with_score()` | Documents + score | You want to inspect/rank retrieval quality |


#### similarity_search() tells us WHAT documents are relevant.
    
#### similarity_search_with_score() tells us WHAT documents are relevant AND gives us a distance score showing how close they are to the query.

### What is a Retriever?

    A Retriever is an object whose job is:

    "Given a question, find the most relevant documents from my knowledge base."

### as_retriever() 

converts the vector store into a LangChain Retriever that can automatically fetch relevant documents for a user query.
    